In [5]:
import torch
import torch.nn as nn
from torchvision import models
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_DIR = os.path.expanduser('~/batch-hackathon')
MODEL_DIR = os.path.join(BASE_DIR, 'models')

def build_resnet18(num_classes=2, freeze_backbone=True):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    in_features = model.fc.in_features
    # ✅ Same Sequential head as Phase 1
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(256, num_classes)
    )

    return model.to(device)

# Load checkpoint
best_model_path = os.path.join(MODEL_DIR, "resnet18_subset_best.pt")
model = build_resnet18(num_classes=2, freeze_backbone=True)
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()

print("Phase 2 model loaded successfully.")

Phase 2 model loaded successfully.


In [6]:
import torch.nn.functional as F

def predict_with_confidence(model, image):
    """
    image shape: [C, H, W]
    Returns: predicted_class (int), confidence (float)
    """
    model.eval()

    with torch.no_grad():
        image = image.unsqueeze(0).to(device)  # Add batch dimension
        outputs = model(image)
        probs = F.softmax(outputs, dim=1)
        conf, pred = torch.max(probs, dim=1)

    return pred.item(), conf.item()

In [8]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import pandas as pd

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [10]:
BASE_DIR = os.path.expanduser('~/batch-hackathon')
SAMPLE_META = os.path.join(BASE_DIR, 'sample_metadata.csv')
MODEL_DIR = os.path.join(BASE_DIR, 'models')
best_model_path = os.path.join(MODEL_DIR, "resnet18_subset_best.pt")

In [11]:
class CIFAKEDataset(Dataset):
    def __init__(self, csv_file, root_dir, split='train', transform=None):
        self.df = pd.read_csv(csv_file)
        self.df = self.df[self.df['split'] == split].reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.df.iloc[idx]['filepath'])
        image = Image.open(img_path).convert('RGB')
        label = int(self.df.iloc[idx]['label'])
        if self.transform:
            image = self.transform(image)
        return image, label

In [12]:
CIFAR_MEAN = [0.4914, 0.4822, 0.4465]
CIFAR_STD  = [0.2023, 0.1994, 0.2010]

val_test_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD),
])

def build_loaders(csv_path, batch_size=64):
    test_ds = CIFAKEDataset(csv_path, BASE_DIR, split='test', transform=val_test_transform)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    return test_loader

sample_test = build_loaders(SAMPLE_META)

In [13]:
def build_resnet18(num_classes=2, freeze_backbone=True):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(256, num_classes)
    )
    return model.to(device)

model = build_resnet18()
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [14]:
def get_high_confidence_fakes(model, dataloader, threshold=0.9):
    selected_images = []

    model.eval()
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            confs, preds = torch.max(probs, dim=1)

            for i in range(len(images)):
                if preds[i].item() == 1 and confs[i].item() > threshold:
                    selected_images.append(images[i].cpu())

    return selected_images

high_conf_fakes = get_high_confidence_fakes(model, sample_test, threshold=0.9)
print(f"Selected {len(high_conf_fakes)} high-confidence fake samples.")

Selected 6 high-confidence fake samples.


In [15]:
def predict_with_confidence(model, image):
    model.eval()
    with torch.no_grad():
        img = image.unsqueeze(0).to(device)
        outputs = model(img)
        probs = torch.softmax(outputs, dim=1)
        conf, pred = torch.max(probs, dim=1)
    return pred.item(), conf.item()

In [16]:
from torchvision.transforms import GaussianBlur

blur_transform = GaussianBlur(kernel_size=3)

def apply_gaussian_blur(image):
    return blur_transform(image)

In [17]:
def apply_frequency_mask(image, mask_ratio=0.3):
    fft = torch.fft.fft2(image)
    fft_shift = torch.fft.fftshift(fft)

    c, h, w = image.shape
    mask = torch.zeros_like(fft_shift)
    center_h, center_w = h // 2, w // 2
    radius_h = int(h * mask_ratio)
    radius_w = int(w * mask_ratio)

    mask[:, center_h-radius_h:center_h+radius_h,
            center_w-radius_w:center_w+radius_w] = 1

    fft_filtered = fft_shift * mask
    fft_ishift = torch.fft.ifftshift(fft_filtered)
    img_back = torch.fft.ifft2(fft_ishift)
    return img_back.real

In [18]:
def run_evasion_experiment(model, image):

    results = []

    # Original
    pred, conf = predict_with_confidence(model, image)
    results.append(('Original', conf, pred))

    # Gaussian Blur
    blurred = apply_gaussian_blur(image)
    pred, conf = predict_with_confidence(model, blurred)
    results.append(('Gaussian Blur', conf, pred))

    # Frequency Mask
    freq_mod = apply_frequency_mask(image)
    pred, conf = predict_with_confidence(model, freq_mod)
    results.append(('Frequency Mask', conf, pred))

    # Combined
    combined = apply_frequency_mask(blurred)
    pred, conf = predict_with_confidence(model, combined)
    results.append(('Final Modified', conf, pred))

    # Print results in a clean format
    for step, conf, pred in results:
        label = 'Fake' if pred==1 else 'Real'
        print(f"{step} → {conf:.2f} {label}")

    return results

In [19]:
for i in range(min(3, len(high_conf_fakes))):
    print(f"\n===== Sample {i+1} =====")
    run_evasion_experiment(model, high_conf_fakes[i])


===== Sample 1 =====
Original → 0.92 Fake
Gaussian Blur → 0.50 Fake
Frequency Mask → 0.61 Fake
Final Modified → 0.51 Fake

===== Sample 2 =====
Original → 0.95 Fake
Gaussian Blur → 0.69 Fake
Frequency Mask → 0.81 Fake
Final Modified → 0.75 Fake

===== Sample 3 =====
Original → 0.97 Fake
Gaussian Blur → 0.60 Fake
Frequency Mask → 0.81 Fake
Final Modified → 0.51 Real
